# Pruning YOLOv8n — Bien Bao VN
**Chuan bi:** baseline.pt + archive/ da co tren Drive tu buoc train.

Cau truc Drive:
```
My Drive/
  archive/ (images, labels, classes.txt, split_dataset/)
  checkpoints/baseline.pt
```

In [ ]:
# CELL 1 - GPU + Drive
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO - doi runtime!')
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 2 - Cai thu vien
!pip install ultralytics -q
print('OK')

In [ ]:
# CELL 3 - Duong dan
from pathlib import Path
DRIVE    = Path('/content/drive/MyDrive')
ARCHIVE  = DRIVE / 'archive'
BASELINE = DRIVE / 'checkpoints' / 'baseline.pt'
OUT_DIR  = DRIVE / 'checkpoints'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('baseline.pt:', BASELINE.exists(), f'{BASELINE.stat().st_size/1024**2:.1f}MB' if BASELINE.exists() else '')
print('archive:', ARCHIVE.exists())

In [ ]:
# CELL 4 - Tao YAML dataset
from pathlib import Path
WORK_DIR = Path('/content/work'); WORK_DIR.mkdir(exist_ok=True)
IMGS = ARCHIVE / 'images'
LBLS = ARCHIVE / 'labels'

def make_paths(list_file, out_file):
    lines = Path(list_file).read_text().splitlines()
    valid = [str((IMGS/l).resolve()) for l in lines if l.strip() and (IMGS/l).exists() and (LBLS/(Path(l).stem+'.txt')).exists()]
    Path(out_file).write_text('\n'.join(valid))
    print(f'{Path(out_file).name}: {len(valid)} anh')

make_paths(ARCHIVE/'split_dataset'/'train_files.txt', WORK_DIR/'train.txt')
make_paths(ARCHIVE/'split_dataset'/'test_files.txt',  WORK_DIR/'val.txt')

names = [l.strip() for l in (ARCHIVE/'classes.txt').read_text().splitlines() if l.strip()]
yaml  = f'train: {WORK_DIR}/train.txt\nval: {WORK_DIR}/val.txt\nnc: {len(names)}\nnames:\n'
for i,n in enumerate(names): yaml += f'  {i}: {n}\n'
YAML = WORK_DIR/'data.yaml'
YAML.write_text(yaml)
print('YAML OK -', len(names), 'classes')

In [ ]:
# CELL 5 - Ham Pruning + Fine-tune
import shutil, time
import numpy as np
import torch.nn as nn
import torch.nn.utils.prune as prune_utils
from ultralytics import YOLO

def run_prune_finetune(baseline_path, amount, yaml_path, out_dir, ft_epochs=10):
    pct = int(amount * 100)
    print(f'\n===== PRUNING {pct}% =====')

    # 1. Load + Prune
    yolo = YOLO(str(baseline_path))
    pt   = yolo.model
    convs = [(n,m) for n,m in pt.named_modules() if isinstance(m, nn.Conv2d)]
    print(f'  Conv2d: {len(convs)} lop')
    for _, m in convs:
        prune_utils.l1_unstructured(m, name='weight', amount=amount)
    total = sum(p.numel() for p in pt.parameters())
    zeros = sum((p==0).sum().item() for p in pt.parameters())
    print(f'  Sparsity: {zeros/total*100:.1f}%')
    for _, m in convs:
        try: prune_utils.remove(m, 'weight')
        except: pass

    # 2. Luu pruned
    p_path = out_dir / f'pruned_{pct}pct.pt'
    yolo.model = pt
    yolo.save(str(p_path))
    print(f'  Pruned: {p_path.name}')

    # 3. Fine-tune
    yolo2 = YOLO(str(p_path))
    yolo2.train(data=str(yaml_path), epochs=ft_epochs, imgsz=640, batch=32,
                lr0=1e-4, lrf=1e-5, patience=100, device=0,
                project='/content/runs', name=f'ft_{pct}', exist_ok=True, plots=False)

    # 4. Luu fine-tuned ve Drive
    best = Path(f'/content/runs/ft_{pct}/weights/best.pt')
    ft_path = out_dir / f'pruned_{pct}pct_finetuned.pt'
    shutil.copy(best, ft_path)
    print(f'  Fine-tuned: {ft_path.name} ({ft_path.stat().st_size/1024**2:.1f}MB)')

    # 5. Evaluate mAP50
    yolo3 = YOLO(str(ft_path))
    val = yolo3.val(data=str(yaml_path), device=0, verbose=False)
    map50 = round(val.box.map50 * 100, 2)
    print(f'  mAP50: {map50}%')
    return ft_path, map50

def benchmark_cpu(model_path, n_runs=30):
    yolo = YOLO(str(model_path))
    dummy = np.zeros((640,640,3), dtype=np.uint8)
    for _ in range(5): yolo.predict(dummy, device='cpu', verbose=False)
    lats = []
    for _ in range(n_runs):
        t = time.perf_counter()
        yolo.predict(dummy, device='cpu', verbose=False)
        lats.append((time.perf_counter()-t)*1000)
    return round(1000/np.mean(lats),1), round(np.mean(lats),1)

print('Ham san sang!')

In [ ]:
# CELL 6 - CHAY PRUNING
# Doi voi muc da co (20%, 30%): comment dong do ra
# Vi du chi chay 40% va 50%: AMOUNTS = [0.4, 0.5]

AMOUNTS = [0.2, 0.3, 0.4, 0.5]   # <-- sua o day neu can

all_results = []
for amt in AMOUNTS:
    ft_path, map50 = run_prune_finetune(BASELINE, amt, YAML, OUT_DIR, ft_epochs=10)
    fps, ms = benchmark_cpu(ft_path)
    mb = round(ft_path.stat().st_size/1024**2, 1)
    all_results.append({'pct': int(amt*100), 'path': ft_path,
                        'map50': map50, 'fps': fps, 'ms': ms, 'mb': mb})

print('\n' + '='*60)
print('TONG KET')
print(f'{"Model":<30} {"mAP50":>8} {"FPS":>6} {"ms":>6} {"MB":>6}')
print('-'*60)
print(f'{"baseline":<30} {97.75:>7.2f}% {37.0:>5.1f} {27.0:>5.1f} {6.0:>5.1f}')
for r in all_results:
    print(f'{"pruned_"+str(r["pct"])+"pct_finetuned":<30} {r["map50"]:>7.2f}% {r["fps"]:>5.1f} {r["ms"]:>5.1f} {r["mb"]:>5.1f}')

In [ ]:
# CELL 7 - Luu CSV ket qua ve Drive
import csv
CSV = DRIVE / 'checkpoints' / 'pruning_results.csv'
rows = [{'model':'baseline','amount_pct':0,'map50':97.75,'fps_cpu':37.0,'mean_ms':27.0,'size_mb':6.0}]
for r in all_results:
    rows.append({'model':f"pruned_{r['pct']}pct_finetuned",
                 'amount_pct':r['pct'], 'map50':r['map50'],
                 'fps_cpu':r['fps'], 'mean_ms':r['ms'], 'size_mb':r['mb']})
with open(CSV,'w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print(f'CSV: {CSV}')
for r in rows:
    print(f"  {r['model']}: mAP50={r['map50']}%, FPS={r['fps_cpu']}, {r['size_mb']}MB")

## Sau khi xong, Drive se co:
```
checkpoints/
  baseline.pt
  pruned_20pct_finetuned.pt
  pruned_30pct_finetuned.pt
  pruned_40pct_finetuned.pt
  pruned_50pct_finetuned.pt
  pruning_results.csv  <-- gui file nay cho agent
```
Download pruning_results.csv va tat ca *_finetuned.pt ve Mac.